## Storing Data with Pickles

In [1]:
import pickle
import pandas as pd

In [2]:
data = [{"name": "Justin"}]

with open("users.pkl", 'wb') as f:
    pickle.dump(data, f) # add data vao users.pkl

In [3]:
data_from_pkl = pickle.load(open('users.pkl', 'rb')) # pickle.loads -> pickle.load
print(data_from_pkl, type(data_from_pkl))

[{'name': 'Justin'}] <class 'list'>


In [4]:
df = pd.DataFrame(data)
df.head()

,name
0,Justin


In [5]:
df.to_pickle("df_pkl")
df.dtypes

name    object
dtype: object

In [6]:
df2 = pd.read_pickle("df_pkl")
df2.head()

,name
0,Justin


In [7]:
df2.dtypes

name    object
dtype: object

In [8]:
df.to_csv("my_df.csv", index=False)

## SQLAlchemy

SQLAlchmey & Django have a similar object-relational mappings (aka `ORM`) are ways to connect Python to a SQL database.

In [9]:
from dataclasses import dataclass

@dataclass
class Movie:
    name:str = 'Unknown'
    genre:str = 'Action'
    year:int = None
    
# class Movie:
#     name = 'Unknown'
#     genre = 'Action'
    
#     def __init__(self, name='', *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.name = name

In [10]:
movie_obj = Movie(name='Interstellar', genre='Sci-Fi')
movie_obj.name

'Interstellar'

In [11]:
data = [{
    "name": "Interstellar",
    "genre": "Sci-Fi"
},
{
    "name": "The Martian",
    "genre": "Sci-Fi"
},
{
    "name": "Arrival",
    "genre": "Sci-Fi"
}
]

In [12]:
df = pd.DataFrame(data)
df.head()

,name,genre
0,Interstellar,Sci-Fi
1,The Martian,Sci-Fi
2,Arrival,Sci-Fi


In [13]:
import sqlalchemy
from sqlalchemy import Column, Integer, String
from sqlalchemy import create_engine, Column, String, Integer
from sqlalchemy.orm import sessionmaker

In [14]:
engine = create_engine("sqlite:///app.db") # mysql, postgres

In [15]:
Session = sessionmaker(bind=engine)
my_sess = Session()

In [16]:
from sqlalchemy.ext.declarative import declarative_base

Base = declarative_base()

In [17]:
class Movie(Base): # table
    __tablename__ = "movies"
    
    id = Column(Integer, primary_key=True) # auto created for us
    name = Column(String) # 'Unknown' # col
    genre = Column(String) # 'Action' # col
    description = Column(String) # 'Action' # col
    year = Column(Integer, nullable=True) # None # col
    
    
    def __repr__(self):
        return f"<Movie name={self.name}>"
        
# saved instance = row

In [18]:
# Add table(s) to database
Base.metadata.create_all(engine)

In [19]:
movie_obj = Movie(name='Interstellar', genre='Sci-Fi')
print(movie_obj.name)

Interstellar


In [20]:
movie_obj.id

In [21]:
my_sess.add(movie_obj) # prepare to save
my_sess.commit() # save

In [22]:
print(movie_obj.id, movie_obj.description)

18 None


In [23]:
movie_obj2 = Movie(name='The Martian', genre='Sci-Fi')
print(movie_obj2.name, movie_obj2.id)

The Martian None


In [24]:
my_sess.add(movie_obj2)

In [25]:
movie_obj3 = Movie(name='Inception', genre='Sci-Fi')
print(movie_obj3.name, movie_obj3.id)

Inception None


In [26]:
my_sess.add(movie_obj3)

In [27]:
my_sess.commit()

In [28]:
print(movie_obj2.id, movie_obj3.id)

19 20


## CRUD in SQLAlchemy

- **C**reate
- **R**etreive
- **U**pdate
- **D**elete

In [29]:
Session = sessionmaker(bind=engine)
session = Session()

In [30]:
# Create
movie = Movie(name="nguyen thanh dung")
session.add(movie)
session.commit()

In [31]:
# Retreive

# Get 1 item
movie_a = session.query(Movie).get(5)
print(movie_a.id, movie_a.name, movie_a.description)

5 Interstellar ho chi minh ha noi


In [32]:
# List
qs = session.query(Movie).all()
print(qs)

[<Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=Guardians of the Galaxy>, <Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=Guardians of the Galaxy>, <Movie name=nguyen chi thong>, <Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=nguyen chi thong>, <Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=nguyen chi thong>, <Movie name=Interstellar>, <Movie name=The Martian>, <Movie name=Inception>, <Movie name=nguyen thanh dung>]


In [33]:
# List & Filter by Column Value
qs = session.query(Movie).filter_by(name='nguyen thanh dung').all()
qs

[<Movie name=nguyen thanh dung>]

In [34]:
# List & Filter by Column Value Containing Something
qs = session.query(Movie).filter(Movie.name.contains("dung")).all()
qs

[<Movie name=nguyen thanh dung>]

In [35]:
# List & Filter by Column Value Containing Something
my_query = input("What are you looking for?\n") or "Unknown"
qs = session.query(Movie).filter(Movie.name.contains(my_query)).all()
print(qs)

What are you looking for?
dung
[<Movie name=nguyen thanh dung>]


In [36]:
# Update

movie_a = session.query(Movie).get(5)
movie_a.description = "ho chi minh ha noi"
print(movie_a.id, movie_a.name, movie_a.description)
session.commit()

5 Interstellar ho chi minh ha noi


In [37]:
movie_a = session.query(Movie).get(5)
print(movie_a.description)

ho chi minh ha noi


In [38]:
qs = session.query(Movie).filter(Movie.name.contains("dung")).all()
for index, movie_obj in enumerate(qs, 1):
    movie_obj.name = f"nguyen chi thong_{index}"
session.commit()

In [44]:
qs = session.query(Movie).filter(Movie.name.contains("thong")).all()
qs

[<Movie name=nguyen chi thong>,
 <Movie name=nguyen chi thong>,
 <Movie name=nguyen chi thong>,
 <Movie name=nguyen chi thong_1>]

# sqlite3

In [40]:
import sqlite3


conx = sqlite3.connect('app.db')
conx2 = create_engine("sqlite:///app.db")

df = pd.read_sql_query("SELECT * FROM movies", conx2)
df.head()

,id,name,genre,description,year
0,1,Interstellar,Sci-Fi,None,None
1,2,The Martian,Sci-Fi,None,None
2,3,Inception,Sci-Fi,None,None
3,4,Guardians of the Galaxy,None,None,None
4,5,Interstellar,Sci-Fi,ho chi minh ha noi,None
